# 01 - Data Cleaning

This notebook loads, explores, and cleans the 3 Eurostat energy datasets.
Cleaned data is saved to `data/clean/` and loaded into the SQLite database.

**Cleaning steps:**
1. Load raw CSVs
2. Initial exploration
3. Data quality assessment
4. Drop metadata columns, rename, cast types
5. Handle null values
6. Final summary
7. Save to `data/clean/` + load into SQLite

**Data source:** Eurostat Energy Balance (NRG_BAL_C), 2005–2024, EU27 countries

In [34]:
import pandas as pd
import sqlite3
from IPython.display import display
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.config import RAW_DATA_DIR, CLEAN_DATA_DIR, DB_PATH, RAW_DATASETS, CLEAN_DATASETS, TABLE_NAMES

## Load Raw Data

Three datasets downloaded from Eurostat's Energy Balance (NRG_BAL_C), all in GWh, covering 27 EU countries from 2005 to 2024:
- **energy_dependency** — total imports, exports, and gross available energy (used to calculate dependency rate)
- **fossil_fuels** — natural gas, oil, and coal broken down by imports / exports / production / availability
- **renewables** — hydro, solar, wind, nuclear, and total renewables broken down by production / availability

In [35]:
datasets = {
    'energy_dependency': pd.read_csv(RAW_DATA_DIR / RAW_DATASETS['energy_dependency']),
    'fossil_fuels':      pd.read_csv(RAW_DATA_DIR / RAW_DATASETS['fossil_fuels']),
    'renewables':        pd.read_csv(RAW_DATA_DIR / RAW_DATASETS['renewables']),
}

print('Data loaded successfully')
print(f"Datasets loaded: {len(datasets)}")

Data loaded successfully
Datasets loaded: 3


## Initial Exploration

Examining the structure and contents of each dataset.

In [36]:
for name, df in datasets.items():
    print('-' * 50)
    print(f"\nDataset: {name}")
    print(f"Rows: {len(df)} | Columns: {len(df.columns)}")
    print('Columns:', list(df.columns))
    print('\nFirst 3 rows:')
    display(df.head(3))
print('-' * 50)

--------------------------------------------------

Dataset: energy_dependency
Rows: 1620 | Columns: 11
Columns: ['DATAFLOW', 'LAST UPDATE', 'freq', 'nrg_bal', 'siec', 'unit', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']

First 3 rows:


,DATAFLOW,LAST UPDATE,freq,nrg_bal,siec,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:NRG_BAL_C(1.0),06/03/26 23:00:00,Annual,Exports,Total,Gigawatt-hour,Austria,2005,47084.289,NaN,NaN
1,ESTAT:NRG_BAL_C(1.0),06/03/26 23:00:00,Annual,Exports,Total,Gigawatt-hour,Austria,2006,38614.345,NaN,NaN
2,ESTAT:NRG_BAL_C(1.0),06/03/26 23:00:00,Annual,Exports,Total,Gigawatt-hour,Austria,2007,45452.897,NaN,NaN


--------------------------------------------------

Dataset: fossil_fuels
Rows: 6480 | Columns: 11
Columns: ['DATAFLOW', 'LAST UPDATE', 'freq', 'nrg_bal', 'siec', 'unit', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']

First 3 rows:


,DATAFLOW,LAST UPDATE,freq,nrg_bal,siec,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:NRG_BAL_C(1.0),06/03/26 23:00:00,Annual,Exports,Solid fossil fuels,Gigawatt-hour,Austria,2005,178.549,NaN,NaN
1,ESTAT:NRG_BAL_C(1.0),06/03/26 23:00:00,Annual,Exports,Solid fossil fuels,Gigawatt-hour,Austria,2006,382.906,NaN,NaN
2,ESTAT:NRG_BAL_C(1.0),06/03/26 23:00:00,Annual,Exports,Solid fossil fuels,Gigawatt-hour,Austria,2007,192.952,NaN,NaN


--------------------------------------------------

Dataset: renewables
Rows: 5400 | Columns: 11
Columns: ['DATAFLOW', 'LAST UPDATE', 'freq', 'nrg_bal', 'siec', 'unit', 'geo', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']

First 3 rows:


,DATAFLOW,LAST UPDATE,freq,nrg_bal,siec,unit,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:NRG_BAL_C(1.0),06/03/26 23:00:00,Annual,Gross available energy,Nuclear heat,Gigawatt-hour,Austria,2005,0.0,NaN,NaN
1,ESTAT:NRG_BAL_C(1.0),06/03/26 23:00:00,Annual,Gross available energy,Nuclear heat,Gigawatt-hour,Austria,2006,0.0,NaN,NaN
2,ESTAT:NRG_BAL_C(1.0),06/03/26 23:00:00,Annual,Gross available energy,Nuclear heat,Gigawatt-hour,Austria,2007,0.0,NaN,NaN


--------------------------------------------------


## Data Quality Assessment

Checking for duplicates, missing values, and data types across all datasets.

In [37]:
def check_data_quality(name, df):
    print(f"\nDataset: {name}")
    print(f"Total duplicated rows: {df.duplicated().sum()}")
    print(f"Duplicate percentage: {(df.duplicated().sum() / len(df) * 100):.2f}%")
    print('\nColumn summary:')
    summary = pd.DataFrame({
        'Type': df.dtypes,
        'Missing Values': df.isnull().sum(),
        'Missing %': (df.isnull().mean() * 100).round(2),
        'Sample Value': df.iloc[0]
    })
    display(summary)


for name, df in datasets.items():
    check_data_quality(name, df)


Dataset: energy_dependency
Total duplicated rows: 0
Duplicate percentage: 0.00%

Column summary:


,Type,Missing Values,Missing %,Sample Value
DATAFLOW,str,0,0.0,ESTAT:NRG_BAL_C(1.0)
LAST UPDATE,str,0,0.0,06/03/26 23:00:00
freq,str,0,0.0,Annual
nrg_bal,str,0,0.0,Exports
siec,str,0,0.0,Total
unit,str,0,0.0,Gigawatt-hour
geo,str,0,0.0,Austria
TIME_PERIOD,int64,0,0.0,2005
OBS_VALUE,float64,0,0.0,47084.289
OBS_FLAG,float64,1620,100.0,NaN



Dataset: fossil_fuels
Total duplicated rows: 0
Duplicate percentage: 0.00%

Column summary:


,Type,Missing Values,Missing %,Sample Value
DATAFLOW,str,0,0.0,ESTAT:NRG_BAL_C(1.0)
LAST UPDATE,str,0,0.0,06/03/26 23:00:00
freq,str,0,0.0,Annual
nrg_bal,str,0,0.0,Exports
siec,str,0,0.0,Solid fossil fuels
unit,str,0,0.0,Gigawatt-hour
geo,str,0,0.0,Austria
TIME_PERIOD,int64,0,0.0,2005
OBS_VALUE,float64,0,0.0,178.549
OBS_FLAG,float64,6480,100.0,NaN



Dataset: renewables
Total duplicated rows: 0
Duplicate percentage: 0.00%

Column summary:


,Type,Missing Values,Missing %,Sample Value
DATAFLOW,str,0,0.0,ESTAT:NRG_BAL_C(1.0)
LAST UPDATE,str,0,0.0,06/03/26 23:00:00
freq,str,0,0.0,Annual
nrg_bal,str,0,0.0,Gross available energy
siec,str,0,0.0,Nuclear heat
unit,str,0,0.0,Gigawatt-hour
geo,str,0,0.0,Austria
TIME_PERIOD,int64,0,0.0,2005
OBS_VALUE,float64,0,0.0,0.0
OBS_FLAG,float64,5400,100.0,NaN


## Cleaning

**What we do and why:**
- Drop `DATAFLOW`, `LAST UPDATE`, `freq`, `OBS_FLAG`, `CONF_STATUS` — Eurostat metadata, not useful for analysis
- Drop `unit` — always `Gigawatt-hour` across all datasets, carries no analytical value
- Rename columns to snake_case for easier SQL and pandas usage
- Cast `year` to int, `value_gwh` to float
- Strip whitespace from string columns

In [38]:
COLUMNS_TO_DROP = ['DATAFLOW', 'LAST UPDATE', 'freq', 'unit', 'OBS_FLAG', 'CONF_STATUS']

COLUMN_RENAMES = {
    'nrg_bal':     'balance_type',
    'siec':        'energy_source',
    'geo':         'country',
    'TIME_PERIOD': 'year',
    'OBS_VALUE':   'value_gwh',
}


def clean_dataset(df):
    df = df.drop(columns=COLUMNS_TO_DROP)
    df = df.rename(columns=COLUMN_RENAMES)
    df['year'] = df['year'].astype(int)
    df['value_gwh'] = pd.to_numeric(df['value_gwh'], errors='coerce')
    for col in df.select_dtypes(include='str').columns:
        df[col] = df[col].str.strip()
    return df


cleaned = {}
for name, df in datasets.items():
    cleaned[name] = clean_dataset(df.copy())
    print(f"Cleaned: {name} → columns: {list(cleaned[name].columns)}")

Cleaned: energy_dependency → columns: ['balance_type', 'energy_source', 'country', 'year', 'value_gwh']
Cleaned: fossil_fuels → columns: ['balance_type', 'energy_source', 'country', 'year', 'value_gwh']
Cleaned: renewables → columns: ['balance_type', 'energy_source', 'country', 'year', 'value_gwh']


## Null Check on value_gwh

Eurostat sometimes uses special flags for missing observations. Checking if any slipped through.

In [39]:
for name, df in cleaned.items():
    nulls = df['value_gwh'].isnull().sum()
    total = len(df)
    print(f"{name}: {nulls} null values in value_gwh ({nulls/total*100:.2f}%)")
    if nulls > 0:
        print('  Null rows:')
        display(df[df['value_gwh'].isnull()])

energy_dependency: 0 null values in value_gwh (0.00%)
fossil_fuels: 0 null values in value_gwh (0.00%)
renewables: 0 null values in value_gwh (0.00%)


## Unique Values Check

Verifying the key categorical columns look correct after cleaning. Pay attention to the `renewables` dataset — `Renewables and biofuels` is a broad aggregate that already includes `Hydro`, `Solar photovoltaic`, and `Wind`. These must never be summed together in analysis or it will cause double-counting.

In [40]:
for name, df in cleaned.items():
    print(f"\n=== {name} ===")
    for col in ['balance_type', 'energy_source', 'unit']:
        if col in df.columns:
            print(f"  {col}: {sorted(df[col].unique())}")
    print(f"  year range: {df['year'].min()} - {df['year'].max()}")
    print(f"  countries ({df['country'].nunique()}): {sorted(df['country'].unique())}")


=== energy_dependency ===
  balance_type: ['Exports', 'Gross available energy', 'Imports']
  energy_source: ['Total']
  year range: 2005 - 2024
  countries (27): ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden']

=== fossil_fuels ===
  balance_type: ['Exports', 'Gross available energy', 'Imports', 'Primary production']
  energy_source: ['Natural gas', 'Oil and petroleum products (excluding biofuel portion)', 'Solid fossil fuels']
  year range: 2005 - 2024
  countries (27): ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slo

## Cleaned Data Summary

Final state of each dataset — confirming row counts, column names, and zero issues before saving.

In [41]:
print('=' * 50)
print('CLEANED DATA SUMMARY')
print('=' * 50)

for name, df in cleaned.items():
    print(f"\n{name}:")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Duplicates: {df.duplicated().sum()}")
    print(f"  Null values: {df.isnull().sum().sum()}")
    display(df.head(3))

CLEANED DATA SUMMARY

energy_dependency:
  Rows: 1620
  Columns: ['balance_type', 'energy_source', 'country', 'year', 'value_gwh']
  Duplicates: 0
  Null values: 0


,balance_type,energy_source,country,year,value_gwh
0,Exports,Total,Austria,2005,47084.289
1,Exports,Total,Austria,2006,38614.345
2,Exports,Total,Austria,2007,45452.897



fossil_fuels:
  Rows: 6480
  Columns: ['balance_type', 'energy_source', 'country', 'year', 'value_gwh']
  Duplicates: 0
  Null values: 0


,balance_type,energy_source,country,year,value_gwh
0,Exports,Solid fossil fuels,Austria,2005,178.549
1,Exports,Solid fossil fuels,Austria,2006,382.906
2,Exports,Solid fossil fuels,Austria,2007,192.952



renewables:
  Rows: 5400
  Columns: ['balance_type', 'energy_source', 'country', 'year', 'value_gwh']
  Duplicates: 0
  Null values: 0


,balance_type,energy_source,country,year,value_gwh
0,Gross available energy,Nuclear heat,Austria,2005,0.0
1,Gross available energy,Nuclear heat,Austria,2006,0.0
2,Gross available energy,Nuclear heat,Austria,2007,0.0


## Save Cleaned CSVs

Keeping `data/raw/` untouched and writing cleaned versions to `data/clean/` — raw data stays as the source of truth, cleaned data is what all analysis notebooks use.

In [42]:
CLEAN_DATA_DIR.mkdir(exist_ok=True)

for name, df in cleaned.items():
    filename = CLEAN_DATASETS[name]
    df.to_csv(CLEAN_DATA_DIR / filename, index=False)
    print(f"Saved: {filename}")

Saved: energy_dependency_clean.csv
Saved: fossil_fuels_clean.csv
Saved: renewables_clean.csv


## Load into SQLite

Creating the `energy.db` database with one table per dataset.
`if_exists='replace'` means re-running this notebook rebuilds the DB from scratch.

In [43]:
DB_PATH.parent.mkdir(exist_ok=True)

conn = sqlite3.connect(DB_PATH)

for name, df in cleaned.items():
    table = TABLE_NAMES[name]
    df.to_sql(table, conn, if_exists='replace', index=False)
    print(f"Loaded table '{table}': {len(df)} rows")

conn.close()
print('\nDatabase ready:', DB_PATH)

Loaded table 'energy_dependency': 1620 rows
Loaded table 'fossil_fuels': 6480 rows
Loaded table 'renewables': 5400 rows

Database ready: g:\CODE\GITHUB\eu-energy-transition-analysis\database\energy.db


## Verify Database

Quick sanity check — query each table to confirm it loaded correctly.

In [44]:
conn = sqlite3.connect(DB_PATH)

for table in TABLE_NAMES.values():
    df_check = pd.read_sql(f"SELECT * FROM {table} LIMIT 3", conn)
    count = pd.read_sql(f"SELECT COUNT(*) as total FROM {table}", conn)['total'][0]
    print(f"\nTable '{table}' — {count} rows:")
    display(df_check)

conn.close()


Table 'energy_dependency' — 1620 rows:


,balance_type,energy_source,country,year,value_gwh
0,Exports,Total,Austria,2005,47084.289
1,Exports,Total,Austria,2006,38614.345
2,Exports,Total,Austria,2007,45452.897



Table 'fossil_fuels' — 6480 rows:


,balance_type,energy_source,country,year,value_gwh
0,Exports,Solid fossil fuels,Austria,2005,178.549
1,Exports,Solid fossil fuels,Austria,2006,382.906
2,Exports,Solid fossil fuels,Austria,2007,192.952



Table 'renewables' — 5400 rows:


,balance_type,energy_source,country,year,value_gwh
0,Gross available energy,Nuclear heat,Austria,2005,0.0
1,Gross available energy,Nuclear heat,Austria,2006,0.0
2,Gross available energy,Nuclear heat,Austria,2007,0.0


## Summary

All datasets cleaned and loaded. Ready for EDA in notebook 02.

**Cleaning steps performed:**
- Dropped 5 Eurostat metadata columns (`DATAFLOW`, `LAST UPDATE`, `freq`, `OBS_FLAG`, `CONF_STATUS`)
- Renamed all columns to snake_case
- Cast `year` → int, `value_gwh` → float
- Stripped whitespace from string columns
- Verified no duplicates or unexpected nulls

**Outputs:**
- `data/clean/energy_dependency_clean.csv`
- `data/clean/fossil_fuels_clean.csv`
- `data/clean/renewables_clean.csv`
- `database/energy.db` (3 tables)

**Data notes for analysis:**
- Energy dependency *rate* is not directly in the data — it must be calculated as `(Imports - Exports) / Gross Available Energy × 100`
- In the `renewables` table, `Renewables and biofuels` is an aggregate that includes `Hydro`, `Solar photovoltaic`, and `Wind` — use the aggregate for totals, use the individual sources only for breakdowns, never sum them together